In [ ]:
# Imports needed
import pandas as pd
import json
import sklearn
import nltk
import seaborn as sns

ModuleNotFoundError: No module named 'nltk'

In [ ]:
# Load only the first 10,000 rows from the dataset
review_data = pd.read_json('yelp_academic_dataset_review.json', lines=True, nrows=10000)

# Show columns to identify location fields
print("Columns in dataset:", review_data.columns.tolist())

# Display the first 10,000 rows
print(f"Loaded {len(review_data)} rows")
print("\nFirst 10,000 rows of the dataset:")
#review_data

# Load only the first # rows from the dataset
# Get all business data to make sure when merging, there are some guaranteed business ID matches from review data read in
business_data = pd.read_json('yelp_academic_dataset_business.json', lines=True)#, nrows=25)

# Show columns to identify location fields
print("Columns in dataset:", business_data.columns.tolist())

# Display the first # rows
print(f"Loaded {len(business_data)} rows")
print("\nFirst # rows of the dataset:")
#business_data

# Only keep businesses that appear in reviews
business_data = business_data[business_data['business_id'].isin(review_data['business_id'])]

Columns in dataset: ['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date']
Loaded 25 rows

First 25 rows of the dataset:


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15
5,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1,1,2,1,I am a long term frequent customer of this est...,2015-09-23 23:10:31
6,6AxgBCNX_PNTOxmbRSwcKQ,r3zeYsv1XFBRA4dJpL78cw,gmjsEdUsKpj9Xxu6pdjH0g,5,0,2,0,Loved this tour! I grabbed a groupon and the p...,2015-01-03 23:21:18
7,_ZeMknuYdlQcUqng_Im3yg,yfFzsLmaWF2d4Sr0UNbBgg,LHSTtnW3YHCeUkRDGyJOyw,5,2,0,0,Amazingly amazing wings and homemade bleu chee...,2015-08-07 02:29:16
8,ZKvDG2sBvHVdF5oBNUOpAQ,wSTuiTk-sKNdcFyprzZAjg,B5XSoSG3SfvQGtKEGQ1tSQ,3,1,1,0,This easter instead of going to Lopez Lake we ...,2016-03-30 22:46:33
9,pUycOfUwM8vqX7KjRRhUEA,59MxRhNVhU9MYndMkz0wtw,gebiRewfieSdtt17PTW6Zg,3,0,0,0,Had a party of 6 here for hibachi. Our waitres...,2016-07-25 07:31:06


## Cuisine Types

In [ ]:
# Only include restaurants (remove doctors, shipping centers, etc.)
business_data = business_data[business_data['categories'].str.contains('Restaurants', na=False)]

def get_cuisine(categories):
    categories = str(categories)

    if 'Chinese' in categories:
        return 'Chinese'
    elif 'Italian' in categories:
        return 'Italian'
    elif 'Mexican' in categories:
        return 'Mexican'
    elif 'French' in categories:
        return 'French'
    elif 'Japanese' in categories or 'Sushi' in categories:
        return 'Japanese'
    elif 'Korean' in categories:
        return 'Korean'
    elif 'Mediterranean' in categories:
        return 'Mediterranean'
    elif 'Vietnamese' in categories:
        return 'Vietnamese'
    elif 'American' in categories or 'Burgers' in categories or 'Fast Food' in categories:
        return 'American'
    else:
        return None

In [ ]:
print("Review rows:", len(review_data))
print("Business rows:", len(business_data))
print(business_data['categories'].head(30))

# Merge two datasets (review, business)

business_data['cuisine_label'] = business_data['categories'].apply(get_cuisine)

merged_data = review_data.merge(business_data[['business_id', 'cuisine_label']], on='business_id')

print("After merge:", len(merged_data))
print("Null labels after merge:", merged_data['cuisine_label'].isna().sum())

# Drop rows with no label
merged_data = merged_data.dropna(subset=['cuisine_label'])
print("After filtering:", len(merged_data))

## Logistic Regression with Unigram

#### Split into training and test sets

In [ ]:
from sklearn.model_selection import train_test_split

print("Dataset size:", len(merged_data))
print(merged_data['cuisine_label'].value_counts())

x_input = merged_data['text']
y_output = merged_data['cuisine_label']

test_size = int(0.2 * len(merged_data))
x_train, x_test, y_train, y_test = train_test_split(x_input, y_output, test_size=test_size, random_state=9)
print(len(x_train), len(x_test))
print(len(y_train), len(y_test))

## Extract Unigram Features

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

unigram_vectorizer = CountVectorizer()
unigram_vectorizer.fit(x_train)
train_features = unigram_vectorizer.transform(x_train)
test_features = unigram_vectorizer.transform(x_test)

print(train_features.shape) # prints (number of rows in the matrix, number of columns)
print(test_features.shape)  # prints (number of rows in the matrix, number of columns)

## Train and Evaluate

In [ ]:
from sklearn.linear_model import LogisticRegression

clf_unigrams = LogisticRegression(max_iter=1000) # Instantiate a logistic regression classifier
clf_unigrams.fit(train_features, y_train) # Train the classifier

In [ ]:
# Evaluate unigram logistic regression classifier
from sklearn.metrics import classification_report # this provides a bunch of useful evaluation metrics

unigram_predictions = clf_unigrams.predict(test_features)

results = pd.DataFrame(classification_report(y_test, unigram_predictions, output_dict=True))
results